In [44]:
!pip install -q bitsandbytes accelerate
!pip install -q git+https://github.com/huggingface/transformers.git@main
!pip install -q git+https://github.com/huggingface/peft.git@main

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [45]:
import pandas as pd
from google.colab import files
import os
filename = "Q2_20230202_majority 1.csv"

if not os.path.exists(filename):
  print(f"'{filename}' not found. Please upload the file.")
  uploaded = files.upload()
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("Q2_20230202_majority 1.csv")
df.sample(5)

,tweet_id,created_at,tweet,label_majority,month
2471,1.415655e+18,2021-07-15 12:49:59+00:00,"jealousy, jealousy? all u need r shades and a vax 😎💉",in-favor,21-Jul
5242,1.491675e+18,2022-02-10 07:25:27+00:00,bill gates: the same person who says the world is overpopulated wants to save your life with a vaccine.,against,22-Feb
3406,1.377918e+18,2021-04-02 09:36:49+00:00,"&lt;2&gt; on 1st april, pune had the target to have 1 lakh vaccination jabs but it was able to do only around 56k. why? because they have not made adequate arrangements to have more vaccination centres and have more people vaccinated. @authackeray",in-favor,21-Apr
4696,1.340017e+18,2020-12-18 19:32:17+00:00,the nurse when she accidentally gives me the moderna vaccine instead of the pfizer one..,in-favor,20-Dec
3076,1.390088e+18,2021-05-05 23:37:29+00:00,"all of this 👇 ..i am happy that general pop is getting vaccinated, but don't think teachers haven't noticed the obvious underhandedness of giving them one whole day to book ahead.",in-favor,21-May


In [46]:
def format_prompt(tweet):

    examples = (
        f'Here are a few examples:'
        f'Tweet: "Vaccines saved my life" Stance: "in-favor"\n'
        f'Tweet: "Still deciding on those shots..." Stance: "neutral-or-unclear"\n'
        f'Tweet: "This vaccine is poison." Stance: "Against"\n'
        f'Now analyze:\n'
    )
    prompt = (
        f'What is the stance of the **author** of the following tweet toward COVID-19 vaccines?\n'
        f'Classify the stance as exactly one of: in-favor, against, neutral-or-unclear. If the stance is unclear or mixed or if the tweet is off-topic, choose "neutral-or-unclear".\n'
        f'{examples}'
        f'Tweet: "{tweet}"\n'
        f'Respond with one exactly of the following options: "in-favor", "against", "neutral-or-unclear".'
    )

    return prompt


df['prompt'] = df['tweet'].apply(format_prompt)
df['target'] = df['label_majority'].str.strip()
print("prompt loaded")

prompt loaded


In [47]:
df_phase1 = df[df['target'].isin(['in-favor', 'against'])]

df_phase2 = df[df['target'] == 'neutral-or-unclear']

df_phase3 = pd.concat([
    df[df['target'] == 'in-favor'].sample(n=1000, random_state=1),
    df[df['target'] == 'against'].sample(n=1000, random_state=1),
    df[df['target'] == 'neutral-or-unclear'].sample(n=1000, random_state=1),
])

print("Phases are prepared.")

Phases are prepared.


In [48]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

def tokenize(example):
    inputs = tokenizer(example['prompt'], max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(example['target'], max_length=8, truncation=True, padding="max_length")
    inputs['labels'] = labels.input_ids
    return inputs

print("Tokenizer loaded.")

Tokenizer loaded.


In [49]:
from transformers import AutoModelForSeq2SeqLM
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model


model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large", load_in_8bit=True)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"]
)
model = get_peft_model(model, lora_config)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [50]:
import shutil
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
from datasets import Dataset


def train_phase(df_phase, phase_name, num_epochs, lr):
    print(f"\n📘 Starting Phase: {phase_name} for {num_epochs} epoch(s) — LR: {lr}")

    dataset = Dataset.from_pandas(df_phase[['prompt', 'target']])
    tokenized = dataset.map(tokenize, batched=True)
    split = tokenized.train_test_split(test_size=0.2)
    train_dataset = split['train']
    eval_dataset = split['test']

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results_{phase_name}",
        learning_rate=lr,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=num_epochs,
        # logging_steps=1,
        # eval_steps=250,
        # save_steps=250,
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir=f'./logs_{phase_name}',
        fp16=False,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to="none"
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    print(f"✅ Phase {phase_name} training complete!")

    # Save & download
    model_path = f"./finetuned-flan-t5-vaccine_{phase_name}"
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    shutil.make_archive(model_path, 'zip', model_path)
    files.download(f"{model_path}.zip")

print("Training function ")

Training function 


In [51]:
# train_phase(df_phase1, "phase1_easy", 2, lr=5e-5)

In [54]:
# train_phase(df_phase2, "phase2_neutral", 2, lr=2e-5)


📘 Starting Phase: phase2_neutral for 2 epoch(s) — LR: 2e-05


Map:   0%|          | 0/1040 [00:00<?, ? examples/s]

/tmp/ipython-input-50-4014438074.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autogr

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [52]:
import zipfile
import os

model_dir = "finetuned-flan-t5-vaccine_phase2_neutral"
zip_path = "/content/finetuned-flan-t5-vaccine_phase2_neutral.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(model_dir)

In [53]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel, PeftConfig

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Load base model config from the PEFT adapter
peft_config = PeftConfig.from_pretrained(model_dir)
base_model = AutoModelForSeq2SeqLM.from_pretrained(peft_config.base_model_name_or_path, load_in_8bit=True)

# Prepare model for training with bitsandbytes
from peft import prepare_model_for_kbit_training
base_model = prepare_model_for_kbit_training(base_model)

# Load PEFT (LoRA) adapter on top of the base model
model = PeftModel.from_pretrained(base_model, model_dir)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [ ]:
train_phase(df_phase3, "phase3_mixed", 3, lr=1e-5)

In [ ]:
!nvidia-smi

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# Load trained model and tokenizer
model_path = "finetuned-flan-t5-vaccine_phase3_mixed"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Load your evaluation/test set
df_test = df.sample(5)
df_test["prompt"] = df_test["tweet"].apply(format_prompt)

In [ ]:
def format_prompt(tweet):

    examples = (
        f'Here are a few examples:'
        f'Tweet: "Vaccines saved my life" Stance: "in-favor"\n'
        f'Tweet: "Still deciding on those shots..." Stance: "neutral-or-unclear"\n'
        f'Tweet: "This vaccine is poison." Stance: "Against"\n'
        f'Now analyze:\n'
    )
    prompt = (
        f'What is the stance of the **author** of the following tweet toward COVID-19 vaccines?\n'
        f'Classify the stance as exactly one of: in-favor, against, neutral-or-unclear. If the stance is unclear or mixed or if the tweet is off-topic, choose "neutral-or-unclear".\n'
        f'{examples}'
        f'Tweet: "{tweet}"\n'
        f'Respond with one exactly of the following options: "in-favor", "against", "neutral-or-unclear".'
    )

    return prompt

def normalize(text):
    return text.strip().lower().replace("-", "").replace("_", "").replace(" ", "")


def predict_stance(tweet, tokenizer, model):
    prompt = format_prompt(tweet)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return prediction

In [ ]:
print("functions loaded")

sample_tweets = [
    "I'm really happy I got my COVID-19 vaccine today!",
    "The government forcing vaccines is just wrong.",
    "I heard mixed things about the vaccine, still unsure if I should take it.",
    "trumps operation warp speed"
]

for tweet in sample_tweets:
    prediction = predict_stance(tweet, tokenizer, model)
    print(f"\nTweet: {tweet}")
    print(f"Predicted Sentiment: {prediction}")


In [ ]:
preds = []
labels = []
wrong = []

# Sample 10% of dataset for evaluation
df_test1, _ = train_test_split(df, test_size=0.98, stratify=df['label_majority'], random_state=42)

print("Generating predictions...")
for _, row in tqdm(df_test1.iterrows(), total=len(df_test1)):
    tweet = row['tweet']
    label = row['label_majority']

    prediction = predict_stance(tweet, tokenizer, model)

    if prediction != label:
        wrong.append({
            "tweet": tweet,
            "label_majority": label,
            "prev_pred": prediction
        })
    preds.append(prediction)
    labels.append(label)

df_test1['predicted'] = preds

In [ ]:
# Overall Accuracy
correct = sum(p == l for p, l in zip(preds, labels))
accuracies = [["Total Validation", f"{correct / len(labels):.3f}"]]

# Per-label Accuracy
label_list = ["in-favor", "against", "neutral-or-unclear"]
for label in label_list:
    total_label = labels.count(label)
    if total_label > 0:
        correct_label = sum(p == l for p, l in zip(preds, labels) if l == label)
        accuracy = correct_label / total_label
        accuracies.append([label, f"{accuracy:.3f}"])
    else:
        accuracies.append([label, "N/A"])

acc_df = pd.DataFrame(accuracies, columns=["Label", "Accuracy"])
print(acc_df)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(labels, preds, labels=label_list)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_list)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()